# Summarize Korean MCQ Evaluation Results

This notebook generates a **RESULTS.md** comparison table from completed EvalHub jobs,
in the style of [evaluate-llm-on-korean-dataset](https://github.com/hyogrin/evaluate-llm-on-korean-dataset).

## Workflow

1. Connect to EvalHub API and list completed `ko-mcq-*` jobs
2. Filter jobs by batch (e.g. timestamp suffix) and collect metrics
3. Call `generate_results_md_from_api()` to produce the final markdown
4. Preview the generated report

> **Alternative**: If you have local CSV files, use the cells at the bottom to generate from CSVs.

## Step 1: Configuration

In [7]:
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    with open(env_file) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ[key] = val

NAMESPACE = os.getenv("NAMESPACE", "demo")
EVALHUB_URL = os.getenv("EVALHUB_URL", "https://localhost:8443")
_r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
RESULTS_DIR = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"EvalHub URL:  {EVALHUB_URL}")

Project root: /Users/hyochoi/dev/rhoai-lmeval-builder-lab
Results dir:  /Users/hyochoi/dev/rhoai-lmeval-builder-lab/results
EvalHub URL:  https://localhost:8443


## Step 2: Generate RESULTS.md from EvalHub API

Completed adapter pods cannot be accessed via `oc cp` (Succeeded phase).
Instead, we pull metrics directly from the EvalHub API and generate RESULTS.md.

In [8]:
import re
from evalhub import SyncEvalHubClient

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

MODEL_NAME = os.getenv("MODEL_NAME", "gemma4-e2b")

# List completed korean-mcq jobs
all_jobs = client.jobs.list()
mcq_jobs = [j for j in all_jobs if "ko-mcq-" in j.name and j.effective_state.value == "completed"]
mcq_jobs.sort(key=lambda j: j.name)

print(f"Found {len(mcq_jobs)} completed ko-mcq jobs:")
for j in mcq_jobs:
    print(f"  {j.name}")

Found 5 completed adapter pods
  0b40ca25-bd6d-4258-92ff-2a-d83f5f3f-bf94-4673-89da-ad76d95vwk9p (2026-05-26T08:47:51Z)
  17ee5182-7e83-4f10-bdd6-36-d983a35d-61a6-46a7-ab98-e115751swc9v (2026-05-26T08:47:51Z)
  3d77eb03-e41b-44f8-bf1b-de-09c7c996-7972-4c2e-a583-9468e896c54v (2026-05-26T08:47:51Z)
  7e372259-16f8-44e4-9afb-33-903b6abe-181e-49d5-bf37-ec1fc2fwt2wg (2026-05-26T08:47:51Z)
  f4f634fe-5be4-4d93-9e67-64-6a52da3e-3d87-4a40-8ec9-5955098pmsqs (2026-05-26T08:47:50Z)


In [16]:
def get_job_metrics(job, client) -> dict | None:
    """Extract metrics and sample count from a completed EvalHub job."""
    status = client.jobs.get(job.id)
    if not status.results or not status.results.benchmarks:
        return None

    bench = status.results.benchmarks[0]
    metrics = bench.metrics if bench.metrics else {}

    # Get sample count from adapter pod logs
    pod_r = subprocess.run(
        ["oc", "get", "pods", "-n", NAMESPACE, "--no-headers",
         "-o", "custom-columns=NAME:.metadata.name"],
        capture_output=True, text=True,
    )
    pods = [p for p in pod_r.stdout.strip().split("\n") if p.startswith(job.id[:8])]
    n_samples = 0
    if pods:
        log_r = subprocess.run(
            ["oc", "logs", "-n", NAMESPACE, pods[0], "-c", "adapter"],
            capture_output=True, text=True,
        )
        m = re.search(r"Examples: (\d+)", log_r.stdout)
        if m:
            n_samples = int(m.group(1))

    return {
        "model": MODEL_NAME,
        "overall_accuracy": metrics.get("overall_accuracy", 0),
        "num_samples": n_samples,
        "metrics": metrics,
    }

### Filter and confirm target jobs

Set `JOB_FILTER` to match the job names you want to include (e.g. `may26-1647`).
**Review the list below before proceeding to the next cell.**

In [ ]:
JOB_FILTER = "may26-1647"  # Change this to match your job batch

selected_jobs = [j for j in mcq_jobs if JOB_FILTER in j.name]
print(f"Found {len(selected_jobs)} jobs matching '{JOB_FILTER}':\n")
print(f"  {'No.':<4} {'Job Name':<50} {'Dataset'}")
print(f"  {'─'*4} {'─'*50} {'─'*20}")
for i, j in enumerate(selected_jobs, 1):
    dataset = j.name.replace("ko-mcq-", "").split("-2000")[0].replace("-", "_")
    print(f"  {i:<4} {j.name:<50} {dataset}")

print(f"\n✓ Confirm the jobs above, then run the next cell to collect metrics.")

No jobs configured for download. Edit JOBS_TO_DOWNLOAD above.
Alternatively, place CSV files manually in results/<dataset>/<model>.csv


### Collect metrics from selected jobs

In [ ]:
job_metrics = {}
for j in selected_jobs:
    dataset = j.name.replace("ko-mcq-", "").split("-2000")[0].replace("-", "_")
    print(f"  {j.name} -> dataset={dataset}")
    info = get_job_metrics(j, client)
    if info:
        job_metrics[dataset] = info
        print(f"    accuracy={info['overall_accuracy']}%, samples={info['num_samples']}")
    else:
        print(f"    No results available")

print(f"\nCollected metrics for {len(job_metrics)} datasets")

## Step 3: Generate RESULTS.md from API Metrics

Uses `generate_results_md_from_api()` to build the report directly from collected metrics.

In [11]:
from utils import generate_results_md_from_api

if job_metrics:
    output_path = RESULTS_DIR / "RESULTS.md"
    md_content = generate_results_md_from_api(job_metrics, output_path)
    print(f"Generated {output_path} ({len(md_content)} chars)")
else:
    print("No job metrics collected. Check the JOB_FILTER and re-run Step 2.")

Available result CSVs:

  click/
    - gemma4-e2b.csv

  haerae/
    - gemma4-e2b.csv

  kmmlu/
    - gemma4-e2b.csv

  kobest_boolq/
    - gemma4-e2b.csv


In [15]:
from IPython.display import Markdown, display

output_path = RESULTS_DIR / "RESULTS.md"
if output_path.exists():
    display(Markdown(output_path.read_text()))
else:
    print("No RESULTS.md generated yet. Check Step 3.")

Overall Accuracy: 72.0% (50 samples)

Category Accuracy:
             category  accuracy
    General Knowledge     75.00
              History     66.67
           Rare Words     66.67
Reading Comprehension     66.67
Standard Nomenclature     81.82


---

## Alternative: Generate from Local CSVs

If you have local CSV files (e.g. from manual placement or a previous run),
you can generate `RESULTS.md` from them instead of the API.

In [ ]:
from utils import generate_results_md

csv_output = RESULTS_DIR / "RESULTS.md"
csv_md = generate_results_md(RESULTS_DIR, csv_output)
print(f"Generated {csv_output} ({len(csv_md)} chars)")

### CSV Directory Structure

Place CSVs in `results/<dataset>/<model>.csv`, then run the cell above.

In [ ]:
if csv_output.exists():
    display(Markdown(csv_output.read_text()))
else:
    print("No RESULTS.md from CSVs. Place CSV files first.")

---

## Reference: CSV Format

If placing CSVs manually, each file must have columns:

`index, answer, pred, response, correct, category`

```
results/
├── click/
│   └── gemma4-e2b.csv
├── haerae/
│   └── gemma4-e2b.csv
├── kmmlu/
│   └── gemma4-e2b.csv
├── kmmlu_hard/
│   └── gemma4-e2b.csv
└── kobest_boolq/
    └── gemma4-e2b.csv
```